# 第 47 章：Capstone 课程证据汇总与教师交接包

这个 notebook 用一个极小的 instructor-handoff table，对比“只按下一轮优先级排序”的 naive baseline 和“先检查开放风险、责任人、证据摘要与归档链接”的教师交接 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_instructor_handoff_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['next_run_priority_score'] = float(row['next_run_priority_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} instructor-handoff items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Recipient role:', ordered_counts(rows, 'recipient_role'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Unresolved risk:', ordered_counts(rows, 'unresolved_risk_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['handoff_id']
    for row in rows
    if row['reference_route'] == 'handoff_ready'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['next_run_priority_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'handoff_ready'
    for row in top4_priority
)
baseline_blocker_hits = sum(
    row['reference_route'] in {'resolve_risk_before_handoff', 'assign_owner_before_handoff'}
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_blocker_intrusion = baseline_blocker_hits / len(top4_priority)

print('Reference handoff-ready items:', reference_ready_ids)
print('Top-4 by next-run priority:')
for row in top4_priority:
    print(
        f"  {row['handoff_id']}: priority={row['next_run_priority_score']:.2f}, "
        f"route={row['reference_route']}, area={row['handoff_area']}"
    )
print('Baseline handoff precision:', round(baseline_ready_precision, 3))
print('Baseline blocker intrusion:', round(baseline_blocker_intrusion, 3))


In [ ]:
def route_handoff_item(row):
    if row['unresolved_risk_status'] in {'open', 'high'}:
        return 'resolve_risk_before_handoff', 'open or high course risk must be handled before handoff'
    if row['owner_status'] != 'assigned':
        return 'assign_owner_before_handoff', 'next action has no clear owner yet'
    if row['evidence_pack_status'] != 'complete':
        return 'complete_evidence_digest', 'course-run evidence pack is not complete enough'
    if row['archive_link_status'] != 'complete':
        return 'complete_evidence_digest', 'archive link or handoff pointer is not complete enough'
    return 'handoff_ready', 'item has evidence, owner, bounded risk, and complete archive pointer'


routed_rows = []
for row in rows:
    route, reason = route_handoff_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Instructor handoff workflow routes:')
for row in routed_rows:
    print(
        f"  {row['handoff_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'handoff_ready']
evidence_queue = [row for row in routed_rows if row['route'] == 'complete_evidence_digest']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_owner_before_handoff']
risk_queue = [row for row in routed_rows if row['route'] == 'resolve_risk_before_handoff']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'handoff_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Handoff-ready queue:', [row['handoff_id'] for row in ready_queue])
print('Complete-evidence queue:', [row['handoff_id'] for row in evidence_queue])
print('Assign-owner queue:', [row['handoff_id'] for row in owner_queue])
print('Resolve-risk queue:', [row['handoff_id'] for row in risk_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured handoff precision:', round(ready_precision, 3))


In [ ]:
def handoff_steps(row):
    steps = []
    if row['unresolved_risk_status'] in {'open', 'high'}:
        steps.append('先解决开放或高风险项，写清当前状态、判断依据和下一步。')
    if row['owner_status'] != 'assigned':
        steps.append('指定下一轮 owner，并记录负责范围和截止时间。')
    if row['evidence_pack_status'] != 'complete':
        steps.append('补齐课程运行证据：日志、学生反馈、助教记录或失败样本。')
    if row['archive_link_status'] != 'complete':
        steps.append('补完整归档链接、版本号和材料位置说明。')
    if row['unresolved_risk_status'] == 'monitored':
        steps.append('保留 monitored caveat，但可以随 ready 项一起交接。')
    return steps or ['可以进入 handoff package；交接时保留版本号和最后更新日期。']


for row in routed_rows:
    if row['route'] != 'handoff_ready':
        print(f"\n{row['handoff_id']} -> {row['route']} ({row['handoff_area']})")
        for step in handoff_steps(row):
            print(' -', step)


In [ ]:
handoff_outline = [
    'One-page course map: calendar, checkpoints, and non-negotiable gates',
    'Runtime and environment digest: known failures, paths, dependencies, and timing',
    'TA handoff: rubric anchors, norming cases, feedback examples, and escalation path',
    'Student feedback digest: common blockers and handout edits before next run',
    'Archive and policy boundary: public, internal, restricted, and hold items',
    'Owner table: next-run tasks, owners, due dates, and unresolved risks',
]

handoff_assistant_prompt = '''你是我的 capstone 教师交接助手。

任务：
1. 先阅读 instructor-handoff table，不要只按 next-run priority 排序；
2. 先检查 unresolved risk；
3. 再检查 owner、evidence pack 和 archive link；
4. 把每个模块 route 到 handoff_ready、complete_evidence_digest、
   assign_owner_before_handoff 或 resolve_risk_before_handoff；
5. 对每个非 ready 模块输出交接前 checklist。

输出格式：
- Handoff-ready modules
- Complete-evidence modules
- Assign-owner modules
- Resolve-risk modules
- Handoff checklist by module
'''

print('Instructor handoff outline:')
for item in handoff_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(handoff_assistant_prompt)


In [ ]:
handoff_snapshot = {
    'dataset': DATA_PATH.name,
    'n_handoff_items': len(rows),
    'baseline_top4_handoff_precision': round(baseline_ready_precision, 3),
    'baseline_blocker_intrusion': round(baseline_blocker_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'handoff_ready': [row['handoff_id'] for row in ready_queue],
    'complete_evidence_digest': [row['handoff_id'] for row in evidence_queue],
    'assign_owner_before_handoff': [row['handoff_id'] for row in owner_queue],
    'resolve_risk_before_handoff': [row['handoff_id'] for row in risk_queue],
    'python_version': platform.python_version(),
}

print('Instructor handoff snapshot:')
for key, value in handoff_snapshot.items():
    print(f'  {key}: {value}')
